# **Build an AI Math Assistant with LangChain Tool Calling**

Estimated time needed: **45** minutes

In this lab, you will learn how to build a simple agent with LangChain, enabling AI agents to perform specific tasks. You'll create a mathematical toolkit that allows an AI agent to perform basic arithmetic operations through natural language interaction. 

Through this lab, you'll build an agent that can understand and solve mathematical queries like "add 25 and 15, then multiply by 2" by breaking down complex operations into simple steps.



# Table of Contents

1. [Objectives](#Objectives)
2. [Setup](#Setup)
3. [Installing required libraries](#Installing-required-libraries)
4. [Loading the LLM: Choosing the right language model](#Loading-the-LLM:-Choosing-the-right-language-model)
5. [Function](#Function)
   - 5.1 [Tool](#Tool)
   - 5.2 [create_agent](#create_agent)
6. [Relationship between agent and LLM](#Relationship-between-agent-and-LLM)
7. [Key parameters of create_agent](#Key-parameters-of-create_agent)
8. [Orchestrating multiple tools with an agent: Mathematical toolkit](#Orchestrating-multiple-tools-with-an-agent:-Mathematical-toolkit)
   - 8.1 [Subtraction tool](#Subtraction-tool)
9. [Building the agent](#Building-the-agent)
   - 9.1 [Exploring LangChain's built-in tools](#Exploring-LangChain's-built-in-tools)
   - 9.2 [Popular built-in tools](#Popular-built-in-tools)
   - 9.3 [Example: Using the Wikipedia tool](#Example:-Using-the-Wikipedia-tool)
10. [Exercise: Create a power tool to calculate exponents](#Exercise:-Create-a-power-tool-to-calculate-exponents)
    - 10.1 [Objective](#Objective)
    - 10.2 [Step 1: Create the power tool](#Step-1:-Create-the-power-tool)
    - 10.3 [Step 2: Create an agent with the power tool](#Step-2:-Create-an-agent-with-the-power-tool)
    - 10.4 [Step 3: Test the agent](#Step-3:-Test-the-agent)
11. [Authors](#authors)


## Objectives

After completing this lab, you will be able to:

- Explain the concept of tools in LangChain
- Create custom tools for specific tasks
- Build an AI agent that can use multiple tools
- Debug and improve tool functionality
- Test tool implementations with various inputs


----


## Setup



For this lab, you will use the following libraries:

- **`langchain`**: For creating AI agents and tools
- **`langchain.chat_models`**: For accessing language models
- **`langchain.agents`**: For creating and managing AI agents

---


## Installing required libraries (local setup with uv on Windows)

This version of the lab runs **locally** and uses **OpenAI only** (no IBM watsonx.ai). Do the setup once in a terminal (PowerShell), not inside the notebook:

```powershell
uv init ai-math-assistant --python 3.12
cd ai-math-assistant
uv add langchain langchain-openai langchain-community wikipedia python-dotenv ipykernel jupyterlab
```

Put this notebook inside the `ai-math-assistant` folder, and create a file named `.env` next to it containing your key:

```
OPENAI_API_KEY=sk-...your key...
```

Then start Jupyter from the project folder:

```powershell
uv run jupyter lab
```

(If you use VS Code instead, open the folder and pick the `.venv` interpreter as the notebook kernel.)

> Note: the original lab used `%pip install ...` cells. Those are removed here because uv manages the packages, and uv's virtual environment doesn't include `pip`.


## Import the required libraries


In [ ]:
import os
import re
import warnings

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

# Loads OPENAI_API_KEY from the .env file in the project folder
load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY not found. Create a .env file with OPENAI_API_KEY=sk-... next to this notebook.")


## Loading the LLM: Choosing the right language model

In this local version of the lab, OpenAI's `ChatOpenAI` is used for **every** agent. (The original lab switched between IBM's Granite model via `ChatWatsonx` and OpenAI; here a single OpenAI model, `llm`, is used throughout.)

Other providers offer LLMs with different strengths:

- **OpenAI (GPT models)**: Versatile, with strong tool-calling support.
- **Meta (LLaMA)**: Open-access, highly customizable for specialized use cases.
- **IBM watsonx Granite**: Aimed at enterprise applications.
- **Anthropic (Claude)**: Focused on safety, reliability, and strong reasoning.
- **Cohere**: Efficient for lightweight, task-specific workloads.

`ChatOpenAI` supports configuration options such as:
- **`temperature`**: Adjusting randomness of responses.
- **`max_tokens`**: Limiting the length of responses.

Change `OPENAI_MODEL` below if you want to use a different OpenAI model.


In [ ]:
OPENAI_MODEL = "gpt-5.4-nano"   # any tool-calling OpenAI chat model works, e.g. "gpt-4o-mini"

llm = ChatOpenAI(model=OPENAI_MODEL)


Let's generate a simple response:


In [ ]:
response = llm.invoke("What is tool calling in langchain?")
print("\nResponse Content: ", response.content)

## Function

In AI, a **tool** will call a basic **function** or capability that can be called on to perform a specific task. Think of it like a single item in a toolbox: just like a hammer, screwdriver, or wrench, an AI toolbox is full of specific functions designed to solve problems or get things done.

When building tools for tool calling, there are a few key principles to keep in mind:

1. **Clear purpose**: Make sure the tool has a well-defined job.
2. **Standardized input**: The tool should accept input in a predictable, structured format so it’s easy to use.
3. **Consistent output**: Always return results in a format that’s easy to process or integrate with other systems.
4. **Comprehensive documentation**: Your tool should include clear, simple documentation that explains what it does, how to use it, and any quirks or limitations.

Remember, documentation isn’t just for other developers—it’s also for the language model (LLM) to understand the tool’s purpose and how to use it effectively.

For this example, you’ll start with a simple tool to add numbers. It’ll check off most of the basic requirements, but one key limitation is that it doesn’t handle **basic error cases**, like ignoring non-numeric input. Improving error handling will make the tool much more robust and ready for real-world use.


In [ ]:
def add_numbers(inputs:str) -> dict:
    """
    Adds a list of numbers provided in the input dictionary or extracts numbers from a string.

    Parameters:
    - inputs (str): 
    string, it should contain numbers that can be extracted and summed.

    Returns:
    - dict: A dictionary with a single key "result" containing the sum of the numbers.

    Example Input (Dictionary):
    {"numbers": [10, 20, 30]}

    Example Input (String):
    "Add the numbers 10, 20, and 30."

    Example Output:
    {"result": 60}
    """
    numbers = [int(x) for x in inputs.replace(",", "").split() if x.isdigit()]

    
    result = sum(numbers)
    return {"result": result}

Directly testing a tool allows you to pinpoint where the problem lies—whether it’s in the tool’s logic, input parsing, or output formatting.
Here, you'll input the string `1 2` and get the sum.


In [ ]:
add_numbers("1 2")


## Tool
The `Tool` class in LangChain serves as a structured wrapper that converts regular Python functions into agent-compatible tools. Each tool needs three key components:
1. A name that identifies the tool
2. The function that performs the actual operation
3. A description that helps the agent understand when to use the tool

Testing section improvement:




In [ ]:
from langchain_core.tools import Tool
add_tool=Tool(
        name="AddTool",
        func=add_numbers,
        description="Adds a list of numbers and returns the result.")

print("tool object",add_tool)

Let's see the parameters of the object:

- **`name`** (*str*):
  - A unique identifier for the tool.

  - **Example**: `"AddTool"`

- **`.invoke`** (*Callable*):
  - The function that the tool wraps.
  - **Example**: `add_numbers`

- **`description`** (*str*):
  - A concise explanation of what the tool does.
  - **Example**: `"Adds a list of numbers and returns the result."`




These attributes allow you to inspect the tool object


In [ ]:
# Tool name
print("Tool Name:")
print(add_tool.name)

# Tool description
print("Tool Description:")
print(add_tool.description)

# Tool function
print("Tool Function:")
print(add_tool.invoke)


You can call the tool's function via the ```add_tool``` object:


In [ ]:
print("Calling Tool Function:")
test_input = "10 20 30 a b" 
print(add_tool.invoke(test_input))  # Example

Testing the tool object ensures:

1. **The tool is correctly set up**:
   - Metadata (`name`, `description`, etc.) is properly defined and aligns with its purpose.
   - The function and schema (if applicable) are correctly configured.

2. **The wrapped function behaves as expected**:
   - The function performs the intended task correctly.
   - It handles edge cases and invalid inputs gracefully.

3. **The tool integrates smoothly with agents**:
   - The tool's output aligns with what the agent expects.
   - There are no compatibility issues when the agent calls the tool.


### `@tool` operator

Now you know how to create a tool with a `Tool` class (using Tool Interface), there's actually another way that you can create a tool using `@tool` decorator. The recommended way to create tools is using the `@tool` decorator. This decorator is designed to simplify the process of tool creation and should be used in most cases. After defining a function, you can decorate it with `@tool` to create a tool that implements the Tool Interface.

`@tool` opertor makes tools out of functions. See below:


In [ ]:
from langchain_core.tools import tool
import re

@tool
def add_numbers(inputs: str) -> dict:
    """Adds all numeric values found in the input string."""
    numbers = [float(num) for num in re.findall(r"-?\d+(?:\.\d+)?", inputs)]
    return {"result": sum(numbers)}

The above function will now act as a tool. You can inspect the tool's schema and other properties using:


In [ ]:
print("Name: \n", add_numbers.name)
print("Description: \n", add_numbers.description) 
print("Args: \n", add_numbers.args) 


You can call the tool using the ```invoke``` method.


In [ ]:
test_input = "what is the sum between 10, 20 and 30 " 
print(add_numbers.invoke(test_input))  # Example


### Use @tool-StructuredTool

The @tool decorator creates a StructuredTool with schema information extracted from function signatures and docstrings as show here. This helps LLMs better understand what inputs the tool expects and how to use it properly. While both approaches work, @tool is generally preferred for modern LangChain applications, especially with LangGraph and function-calling models.


In [ ]:
# Comparing the two approaches
print("Tool Constructor Approach:")

print(f"Has Schema: {hasattr(add_tool, 'args_schema')}")
print("\n")

print("@tool Decorator Approach:")


print(f"Has Schema: {hasattr(add_numbers, 'args_schema')}")
print(f"Args Schema Info: {add_numbers.args}")

In this example, the tool has two inputs: a string containing the numbers to add, and a second boolean input that determines whether to sum the absolute values of those numbers.


In [ ]:
from typing import List

@tool
def add_numbers_with_options(numbers: List[float], absolute: bool = False) -> float:
    """
    Adds a list of numbers provided as input.

    Parameters:
    - numbers (List[float]): A list of numbers to be summed.
    - absolute (bool): If True, use the absolute values of the numbers before summing.

    Returns:
    - float: The total sum of the numbers.
    """
    if absolute:
        numbers = [abs(n) for n in numbers]
    return sum(numbers)

Let's compare the arguments for add_numbers_with_options and add_numbers. Both are structured tools. They both include the inputs field, which is a string input. However, add_numbers_with_options has an additional key-value pair: absolute, a boolean field with a default value of False. This means add_numbers_with_options supports optional behavior—taking the absolute value of the numbers—while add_numbers only handles basic numeric extraction and summation


In [ ]:
print(f"Args Schema Info: {add_numbers_with_options.args}")
print(f"Args Schema Info: {add_numbers.args}")

You can call the tool using a dictionary as input, where each key corresponds to a parameter name and the value is the input for that parameter. For example, to control whether the numbers are summed normally or with absolute values, set the absolute flag to False or True: You get -6 and 6, respectively 


In [ ]:
print(add_numbers_with_options.invoke({"numbers":[-1.1,-2.1,-3.0],"absolute":False}))
print(add_numbers_with_options.invoke({"numbers":[-1.1,-2.1,-3.0],"absolute":True}))

## Improved tool return types with Python typing

When creating tools, you must accurately specify their return values. This helps the agent understand and handle different possible outputs.



The function `sum_numbers_with_complex_output` returns a more flexible output format. It returns a dictionary containing a float value when numbers are successfully summed, or a descriptive error message as a string if no numbers are found or an issue occurs during processing.


In [ ]:
from typing import Dict, Union

@tool
def sum_numbers_with_complex_output(inputs: str) -> Dict[str, Union[float, str]]:
    """
    Extracts and sums all integers and decimal numbers from the input string.

    Parameters:
    - inputs (str): A string that may contain numeric values.

    Returns:
    - dict: A dictionary with the key "result". If numbers are found, the value is their sum (float). 
            If no numbers are found or an error occurs, the value is a corresponding message (str).

    Example Input:
    "Add 10, 20.5, and -3."

    Example Output:
    {"result": 27.5}
    """
    matches = re.findall(r'-?\d+(?:\.\d+)?', inputs)
    if not matches:
        return {"result": "No numbers found in input."}
    try:
        numbers = [float(num) for num in matches]
        total = sum(numbers)
        return {"result": total}
    except Exception as e:
        return {"result": f"Error during summation: {str(e)}"}

The function `sum_numbers_from_text` returns a straightforward output format. It extracts all integer values from the input string, sums them, and returns the total as a float. This function assumes that at least one valid number is present in the input and does not handle cases where no numbers are found or where an error might occur.


In [ ]:
@tool
def sum_numbers_from_text(inputs: str) -> float:
    """Adds all numeric values found in the input string."""
    numbers = [float(num) for num in re.findall(r"-?\d+(?:\.\d+)?", inputs)]
    return sum(numbers)

### `create_agent`

When you set up an agent, you're connecting tools and a language model to work together. The agent uses the model to understand the user's request, decide which tool to call, and use the tool result to produce a response.

#### **Relationship between agent and LLM**
- The **agent** manages the tool-calling workflow.
- The **LLM** interprets the user's input and decides whether a tool is needed.
- The **tools** perform the actual operations.

#### **Key parameters of `create_agent`**

1. **`model`** - the language model used by the agent.
2. **`tools`** - a list of tools available to the agent.
3. **`system_prompt`** - optional instructions that guide the agent's behavior.
4. **`debug`** - optionally enables detailed execution information.


You can now create an agent object using `create_agent`.


In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[add_tool]
)

Now, you can run the agent by asking a question.


> When running an agent using .invoke(), you may occasionally encounter a situation where the code keeps executing indefinitely, even if the LLM has already produced a valid answer. This typically happens when the system encounters an OUTPUT_PARSING_ERROR — often due to formatting issues in the LLM's response.
>
> In such cases, the agent can get stuck in a loop and won’t terminate on its own. If you see this happening, simply click the stop button (■) in the top toolbar to interrupt execution.


In [ ]:
# Use the agent
response = agent.invoke({
    "messages": [("human", "In 2023, the US GDP was approximately $27.72 trillion, while Canada's was around $2.14 trillion and Mexico's was about $1.79 trillion what is the total.")]
})
print(response["messages"][-1].content)

In [ ]:
response

In [ ]:
agent.invoke({"messages": [("human", "Add 10, 20, two and 30")]})

The agent was asked to add the numbers 10, 20, "two," and 30. The agent first noticed that one of the inputs was the word "two" instead of a number, so it converted "two" to its numeric form, which is 2. After preparing the list of numbers (10, 20, 2, and 30), the agent decided to use the `AddTool` to perform the addition. It passed the numbers to the tool, which calculated the sum and returned the result as 62. Finally, the agent provided the answer: **62**.


#### **Structured tool calling with `create_agent`**

Modern LangChain agents use `create_agent` for tool calling. Tools created with the `@tool` decorator can expose structured input schemas and can return structured values. The same agent API can be used with both simple and multi-input tools.


In the agent example below, use `sum_numbers_from_text` as a tool with `create_agent`. For the LLM, you'll use the OpenAI model `llm`.


In [ ]:
agent_2 = create_agent(
    model=llm,
    tools=[sum_numbers_from_text]
)
response = agent_2.invoke({
    "messages": [("human", "Add -10, -20, and -30 using absolute values.")]
})
print(response["messages"][-1].content)

Now, for the below agent, you will be using `sum_numbers_with_complex_output` as the tool, again with the OpenAI model `llm`. The modern `create_agent` API handles the tool-calling workflow without selecting a legacy agent type.


In [ ]:
agent_3 = create_agent(
    model=llm,
    tools=[sum_numbers_with_complex_output]
)
response = agent_3.invoke({"messages": [("human", "Add 10, 20 and 30")]})
print(response["messages"][-1].content)

Now, let's move on to tools with multiple inputs. The agent below uses the OpenAI model `llm` and `add_numbers_with_options` as the tool, which accepts multiple input parameters. The same `create_agent` API can be used with structured multi-input tools.


In [ ]:
agent_2 = create_agent(
    model=llm,
    tools=[add_numbers_with_options]
)

In [ ]:
response = agent_2.invoke({
    "messages": [("Add -10, -20, and -30 using absolute values.")]
})
print(response)

Here is the same multi-input tool with a fresh agent (in the original lab this compared Granite with OpenAI; here both use OpenAI).


In [ ]:
agent_openai = create_agent(
    model=llm,
    tools=[add_numbers_with_options]
)

In [ ]:
response = agent_openai.invoke({
    "messages": [("human", "Add -10, -20, and -30 using absolute values.")]
})
print(response)

### **`create_agent`**

`create_agent` is the modern LangChain API for building agents that use chat models and tools. It creates an agent graph that repeatedly calls the model and executes requested tools until the task is complete.

#### **Key parameters of `create_agent`**

1. **`model`**
   - The language model that powers the agent.

2. **`tools`**
   - A list of tools the agent can use.

3. **`system_prompt` (optional)**
   - Instructions that guide the agent's behavior.

#### How it works

1. The user message enters the agent.
2. The model decides whether a tool call is needed.
3. Requested tools are executed.
4. Tool results are added to the message history.
5. The model produces the final response when no further tool calls are needed.

See the [LangChain `create_agent` documentation](https://reference.langchain.com/python/langchain/agents/factory/create_agent).


In [ ]:
from langchain.agents import create_agent

agent_exec = create_agent(model=llm, tools=[sum_numbers_from_text])
msgs = agent_exec.invoke({"messages": [("human", "Add the numbers -10, -20, -30")]})

In [ ]:
print(msgs["messages"][-1].content)

## Orchestrating multiple tools with an agent: Mathematical toolkit
In real-world applications, a single tool is often insufficient to address the complexity and diversity of user requests. Tasks such as data analysis, performing calculations, or retrieving specific information require specialized capabilities that cannot be fulfilled by a single function. By equipping an agent with multiple tools, each tailored to a distinct purpose, you'll create a system that can dynamically select and utilize the appropriate tool based on the user’s query. This approach enhances the flexibility and scalability of the AI, allowing it to handle a broad spectrum of tasks with precision and efficiency. The orchestration of multiple tools ensures that the agent can seamlessly manage complex workflows, making it an essential framework for building robust and versatile AI systems.

To demonstrate this concept, let’s create additional tools, i.e, a mathematical toolkit. In addition to the addition tool, you will now create tools for subtraction, multiplication, and division. These tools will be integrated into an agent capable of handling various mathematical queries, showcasing how multiple tools can work together within a single AI system.

### Subtraction tool
The subtraction tool is designed to take a list of numbers and return the result of subtracting all subsequent numbers from the first number. This tool is particularly useful for handling queries involving differences, such as "What is 100 minus 20 and then minus 10?". 


In [ ]:
@tool
def subtract_numbers(inputs: str) -> dict:
    """
    Extracts numbers from a string and successively subtracts each number after the first
    from the first number.

    Example Input:
    "100, 20, 10"

    Example Output:
    {"result": 70}
    """
    numbers = [float(num) for num in re.findall(r"-?\d+(?:\.\d+)?", inputs)]

    if not numbers:
        return {"result": 0}

    result = numbers[0]
    for num in numbers[1:]:
        result -= num

    return {"result": result}

You can inspect the tool's schema and other properties using:


In [ ]:
print("Name: \n", subtract_numbers.name)
print("Description: \n", subtract_numbers.description) 
print("Args: \n", subtract_numbers.args) 

Let's use it directly by calling the function:


In [ ]:
print("Calling Tool Function:")
test_input = "10 20 30 a b" 
print(subtract_numbers.invoke(test_input))  # Example

Let's now build multiple tools, starting with `MultiplyTool` and `DivideTool`, by defining two functions: `multiply_numbers` and `divide_numbers`. These functions are simple - `multiply_numbers` takes a list of numbers in string format and returns their product, while `divide_numbers` takes the first number and divides it by each subsequent number in sequence. Instead of manually wrapping these functions in the Tool class, you'll use the `@tool` decorator to automatically convert them into LangChain tools, using their docstrings as descriptions. These decorated tools can then be added directly to the agent alongside other operations like addition or subtraction, allowing the agent to intelligently select the appropriate operation based on the user's query, making it versatile for handling various math problems.


In [ ]:
# Multiplication Tool
@tool
def multiply_numbers(inputs: str) -> dict:
    """Extracts numbers from a string and calculates their product."""
    numbers = [float(num) for num in re.findall(r"-?\d+(?:\.\d+)?", inputs)]

    if not numbers:
        return {"result": 1}

    result = 1
    for num in numbers:
        result *= num
    return {"result": result}

In [ ]:
# Division Tool
@tool
def divide_numbers(inputs: str) -> dict:
    """Extracts numbers from a string and divides the first by each subsequent number."""
    numbers = [float(num) for num in re.findall(r"-?\d+(?:\.\d+)?", inputs)]

    if not numbers:
        return {"result": 0}

    if any(num == 0 for num in numbers[1:]):
        return {"result": "Cannot divide by zero."}

    result = numbers[0]
    for num in numbers[1:]:
        result /= num
    return {"result": result}

When testing these mathematical tools directly, notice that using raw string inputs like "2, 3, and 4" or "100, 5, 2" will fail. The tools are designed to work with numeric inputs only - they don't have the natural language understanding that comes with the LLM agent layer. To test properly, you need to use a numeric value:


In [ ]:
# Testing multiply_tool
multiply_test_input = "2, 3, and 4"
multiply_result = multiply_numbers.invoke(multiply_test_input)
print("--- Testing MultiplyTool ---")
print(f"Input: {multiply_test_input}")
print(f"Output: {multiply_result}")

In [ ]:
# Testing divide_tool
divide_test_input = "100, 5, 2"
divide_result = divide_numbers.invoke(divide_test_input)
print("--- Testing DivideTool ---")
print(f"Input: {divide_test_input}")
print(f"Output: {divide_result}")

## Building the agent

With the implementation of mathematical operators—addition, subtraction, multiplication, and division — you have established a simple yet functional mathematical toolkit. Unlike before, the agent must now not only select the appropriate tool and process the input but also determine the correct mathematical operation based on the user's query.

Let's create the agent object. first, combine all tools into a single list:


In [ ]:
tools = [add_numbers,subtract_numbers, multiply_numbers, divide_numbers]
tools

Like before, you will create the agent with the tools and the language model as input.


In [ ]:
# Create the agent with all tools
math_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful mathematical assistant that can perform various operations. Use the tools precisely and explain your reasoning clearly."
)

In [ ]:
response = math_agent.invoke({
    "messages": [("human", "What is 25 divided by 4?")]
})

# Get the final answer
final_answer = response["messages"][-1].content
print(final_answer)

In [ ]:
response_2 = math_agent.invoke({
    "messages": [("human", "Subtract 100, 20, and 10.")]
})

# Get the final answer
final_answer_2 = response_2["messages"][-2].content
print(final_answer_2)

When the agent tries to subtract 20 and 10 from 100, something unexpected happens. The tool called `SubtractTool` works differently than the agent expected. When you type in "100, 20, 10", instead of giving you 70 like you'd expect, it gives you -130. This happens because your special calculator first turns 100 into -100, then subtracts the other numbers.

```The Confusion``` 

The agent expects the function to work like normal math 100 - 20 - 10 = 70). When the agent tries to fix this by breaking the problem into smaller steps, it still gets unexpected answers because the calculator keeps using its special rules.


```Getting Stuck```

 The agent keeps trying the same approach repeatedly, not realizing why it isn't working. Eventually, it runs out of time without solving the problem.
 
 Before you fix the problem, let's test the other tools.


In [ ]:
print("\n--- Testing MultiplyTool ---")
response = math_agent.invoke({
    "messages": [("human", "Multiply 2, 3, and 4.")]
})
print("Agent Response:", response["messages"][-1].content)

print("\n--- Testing DivideTool ---")
response = math_agent.invoke({
    "messages": [("human", "Divide 100 by 5 and then by 2.")]
})
print("Agent Response:", response["messages"][-1].content)

Now lets change the `SubtractTool` so it subtracts the numbers directly (without negating the first number). This aligns the tool’s behavior with standard arithmetic and the agent’s expectations.


In [ ]:
@tool
def new_subtract_numbers(inputs: str) -> dict:
    """
    Extracts numbers from a string and performs subtraction sequentially, starting with the first number.

    This function is designed to handle input in string format, where numbers may be separated by spaces, 
    commas, or other delimiters. It parses the input string, extracts numeric values, and calculates 
    the result by subtracting each subsequent number from the first. inputs[0]-inputs[1]-inputs[2]

    Parameters:
    - inputs (str): 
      A string containing numbers to subtract. The string can include spaces, commas, or other 
      delimiters between the numbers.

    Returns:
    - dict: 
      A dictionary containing the key "result" with the calculated difference as its value. 
      If no valid numbers are found in the input string, the result defaults to 0.

    Example Usage:
    - Input: "100, 20, 10"
    - Output: {"result": 70}

    Limitations:
    - The function does not handle cases where numbers are formatted with decimals or other non-integer representations.
    """
    # Extract numbers from the string
    numbers = [int(num) for num in inputs.replace(",", "").split() if num.isdigit()]

    # If no numbers are found, return 0
    if not numbers:
        return {"result": 0}

    # Start with the first number
    result = numbers[0]

    # Subtract all subsequent numbers
    for num in numbers[1:]:
        result -= num

    return {"result": result}


## Note: Tool naming when demonstrating different approaches

In this lab, two different ways to create the same mathematical tool (addition) are intentionally shown:

1. Using the `Tool()` constructor approach (`add_tool`)
2. Using the `@tool` decorator approach (`add_numbers`)

This is to compare different LangChain tool creation methods for educational purposes. In a production application, you would typically choose one consistent approach rather than having duplicate tools for the same functionality.

When building real agents, duplicate tools with similar functions will confuse the LLM, as it won't know which one to choose. Always use unique tools with clearly differentiated purposes in production code.


Next, create a new agent, ensuring it incorporates the updated subtraction tool.


In [ ]:
tools_updated = [add_numbers, new_subtract_numbers, multiply_numbers, divide_numbers]
# Create the agent with all tools
math_agent_new = create_agent(
    model=llm,
    tools=tools_updated,
    system_prompt="You are a helpful mathematical assistant that can perform various operations. Use the tools precisely and explain your reasoning clearly."
)
print("agent", math_agent_new)

Now, you are going to create a Python dictionary to test multiple use cases for your agent. Testing your agent is important on its own because it helps ensure that it works correctly in different situations. Automating test cases makes this process easier and helps catch errors before they become a problem. A good test suite checks how the agent handles different inputs, including tricky cases like dividing by zero, working with large numbers, and handling decimals. You can also test how the agent deals with mixed operations, like combining addition and multiplication.


In [ ]:
# Test Cases
test_cases = [
    {
        "query": "Subtract 100, 20, and 10.",
        "expected": {"result": 70},
        "description": "Testing subtraction tool with sequential subtraction."
    },
    {
        "query": "Multiply 2, 3, and 4.",
        "expected": {"result": 24},
        "description": "Testing multiplication tool for a list of numbers."
    },
    {
        "query": "Divide 100 by 5 and then by 2.",
        "expected": {"result": 10.0},
        "description": "Testing division tool with sequential division."
    },
    {
        "query": "Subtract 50 from 20.",
        "expected": {"result": -30},
        "description": "Testing subtraction tool with negative results."
    }

]

This code extracts the actual computation result from the agent's response structure. Unlike a direct tool invocation that returns a simple dictionary, LangGraph agents return a complex structure containing the entire conversation history as a list of messages. To find the computation result, you must locate the specific ToolMessage in this list (identified by its name matching one of the math tools), then parse its content, which contains the actual result as a JSON string. This approach is necessary because the result isn't accessible directly from the response object but is instead nested within the message history, requiring you to navigate through the messages to find and extract the relevant data for comparison with your expected values.


In [ ]:
correct_tasks = []
# Corrected test execution
for index, test in enumerate(test_cases, start=1):
    query = test["query"]
    expected_result = test["expected"]["result"]  # Extract just the value
    
    print(f"\n--- Test Case {index}: {test['description']} ---")
    print(f"Query: {query}")
    
    # Properly format the input
    response = math_agent_new.invoke({"messages": [("human", query)]})
    
    # Find the tool message in the response
    tool_message = None
    for msg in response["messages"]:
        if hasattr(msg, 'name') and msg.name in ['add_numbers', 'new_subtract_numbers', 'multiply_numbers', 'divide_numbers']:
            tool_message = msg
            break
    
    if tool_message:
        # Parse the tool result from its content
        import json
        tool_result = json.loads(tool_message.content)["result"]
        print(f"Tool Result: {tool_result}")
        print(f"Expected Result: {expected_result}")
        
        if tool_result == expected_result:
            print(f"✅ Test Passed: {test['description']}")
            correct_tasks.append(test["description"])
        else:
            print(f"❌ Test Failed: {test['description']}")
    else:
        print("❌ No tool was called by the agent")

print("\nCorrectly passed tests:", correct_tasks)

The current functions would benefit from enhanced error handling and input validation. The add_numbers, subtract_numbers, multiply_numbers, and divide_numbers functions should be updated to handle decimal numbers using float conversion, validate inputs more strictly, and provide clear error messages for edge cases. For example, divide_numbers should explicitly check for division by zero, and all functions should gracefully handle non-numeric inputs like "two" or "hundred". The test cases should be expanded beyond basic operations to include edge cases like divided by zero, empty inputs, and mixed numeric/text inputs (e.g., "divide one hundred by 5"). Also consider adding tests for decimal numbers (e.g., "multiply 3.5 by 2") and sequential operations (e.g., "multiply 10 by 2 then add 5"). This comprehensive testing approach ensures the agent can handle a wide range of real-world mathematical queries.


## **Exploring LangChain's built-in tools**

While creating custom tools is powerful, LangChain provides a rich ecosystem of **pre-built tools** that solve common tasks out of the box. These tools abstract away complex implementation details (API calls, input parsing, error handling) and let you focus on building robust agents quickly.


---

#### **Why use built-in tools?**
- **Reliability**: Tested and maintained by the LangChain community.
- **Time-saving**: No need to reinvent the wheel for common tasks.
- **Integration**: Designed to work seamlessly with LangChain agents.

---


#### **Popular built-in tools**
Here are some widely used tools from `langchain_community.tools`:

| Tool Name               | Description                                                                 |
|-------------------------|-----------------------------------------------------------------------------|
| `WikipediaQueryRun`     | Search Wikipedia for factual information.                                   |
| `GoogleSearchRun`       | Perform web searches using Google’s API.                                    |
| `PythonREPLTool`        | Execute Python code in a safe environment.                                  |
| `OpenWeatherMapQueryRun`| Fetch real-time weather data.                                               |
| `YouTubeSearchTool`     | Search for YouTube videos.                                                  |

---



#### **Example: Using the Wikipedia tool**
Let’s enhance the math agent with Wikipedia access to answer questions requiring factual context.


Now, you'll start by creating a Wikipedia search tool using `@tool` operator. This tool will allow the agent to fetch factual information from Wikipedia when needed.


In [ ]:
from langchain_community.utilities import WikipediaAPIWrapper
import wikipedia.wikipedia as wp
import requests as _requests
import time

wp.API_URL = "https://en.wikipedia.org/w/api.php"

def _patched_request(params, _retries=3):
    params['format'] = 'json'
    if 'action' not in params:
        params['action'] = 'query'
    headers = {"User-Agent": "AI-Math-Assistant-Lab/1.0 (contact@example.com)"}

    last_exc = None
    for attempt in range(_retries):
        r = _requests.get(wp.API_URL, params=params, headers=headers)
        try:
            return r.json()
        except ValueError as e:
            last_exc = e
            print(f"Attempt {attempt+1}/{_retries} failed — status {r.status_code}, body: {r.text[:200]!r}")
            time.sleep(1)
    raise last_exc

wp._wiki_request = _patched_request

# Create a Wikipedia tool using the @tool decorator
@tool
def search_wikipedia(query: str) -> str:
    """Search Wikipedia for factual information about a topic.
    
    Parameters:
    - query (str): The topic or question to search for on Wikipedia
    
    Returns:
    - str: A summary of relevant information from Wikipedia
    """
    wiki = WikipediaAPIWrapper()
    return wiki.run(query)

In [ ]:
search_wikipedia.invoke("What is tool calling?")

Now, you will **create a list of available tools** (both custom math tools and a using wikipedia tool `search_wikipedia`) and then **initialize an agent** that can use these tools to solve problems. This agent will combine:  
- Custom math tools (`add_numbers`, `new_subtract_numbers`, etc.) for arithmetic operations.  
- A built-in tool (`wiki_tool`, e.g., for Wikipedia searches) for additional functionality.  

By combining these tools, the agent can handle **both mathematical calculations** (e.g., addition, subtraction) and **informational queries** (e.g., fetching facts from Wikipedia), depending on the user’s request.


In [ ]:
# Update your tools list to include the Wikipedia tool
tools_updated = [add_numbers, new_subtract_numbers, multiply_numbers, divide_numbers, search_wikipedia]

# Create the agent with all tools including Wikipedia
math_agent_updated = create_agent(
    model=llm,
    tools=tools_updated,
    system_prompt="You are a helpful assistant that can perform various mathematical operations and look up information. Use the tools precisely and explain your reasoning clearly."
)

Now, you will **ask the agent a multi-step question** that requires:  
1. **Online searching** (using `search_wikipedia` or another built-in tool) to fetch real-world data.  
2. **Mathematical computation** (using `multiply_numbers`) to process the retrieved data.  


In [ ]:
query = "What is the population of Canada? Multiply it by 0.75"

response = math_agent_updated.invoke({"messages": [("human", query)]})

print("\nMessage sequence:")
for i, msg in enumerate(response["messages"]):
    print(f"\n--- Message {i+1} ---")
    print(f"Type: {type(msg).__name__}")
    if hasattr(msg, 'content'):
        print(f"Content: {msg.content}")
    if hasattr(msg, 'name'):
        print(f"Name: {msg.name}")
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        print(f"Tool calls: {msg.tool_calls}")

**How it works**:
1. The agent first uses `search_wikipedia` to find Canada's population.
2. Extracts the numeric value from Wikipedia’s response.
3. Uses `multiply_numbers` to calculate 75% of the population.
4. Returns the final result with context.


For a full list of available tools, see the [LangChain Tools Documentation](https://python.langchain.com/docs/integrations/tools/).


## **Exercise: Create a power tool to calculate exponents**

#### **Objective**
In this exercise, you will create a custom tool that calculates the power of a number (e.g., \( x^y \)). You will then integrate this tool into an agent and test its functionality.

---

#### **Step 1: Create the power tool**

1. **Define the Tool Function**:
   - Create a Python function named `calculate_power` that takes a string as input. The string will contain two numbers: the base (\( x \)) and the exponent (\( y \)).
   - The function should extract the numbers, calculate \( x^y \), and return the result as a dictionary with the key `"result"`.


In [ ]:
#TODO


<details>
    <summary>Click here for Solution</summary>

```python
def calculate_power(input_text: str) -> dict:
    """
    Calculates the power of a number (x^y).

    Parameters:
    - input_text (str): A string like "2, 3", "2 3", "5^2", or "2 to the power of 3".

    Returns:
    - dict: {"result": <calculated value>} or an error message.
    """
    # Try to extract expressions like "5^2"
    match = re.search(r"(\d+(?:\.\d+)?)\s*\^+\s*(\d+(?:\.\d+)?)", input_text)
    if match:
        base = float(match.group(1))
        exponent = float(match.group(2))
        return {"result": base ** exponent}

    # Try to extract expressions like "2 to the power of 3"
    match = re.search(r"(\d+(?:\.\d+)?)\s*(?:to\s+the\s+power\s+of)\s*(\d+(?:\.\d+)?)", input_text, re.IGNORECASE)
    if match:
        base = float(match.group(1))
        exponent = float(match.group(2))
        return {"result": base ** exponent}

    # Fallback: assume two numbers separated by space or comma
    try:
        numbers = [float(num) for num in input_text.replace(",", " ").split()]
        if len(numbers) != 2:
            return {"result": "Invalid input. Please provide exactly two numbers."}
        base, exponent = numbers
        return {"result": base ** exponent}
    except ValueError:
        return {"result": "Invalid input format. Provide input like '2 3', '2^3', or '2 to the power of 3'."}


```

</details>


2. **Create the tool object**:
   - Use the `@tool` decorator to create a tool from the `calculate_power` function.
   - Provide a clear name and description through the function definition.


In [ ]:
#TODO


<details>
   <summary>Click here for Solution</summary>

```python
from langchain_core.tools import tool

@tool
def power_tool(input_text: str) -> dict:
    """Calculates the power of a number (x^y)."""
    return calculate_power(input_text)
```

</details>


#### **Step 2: Create an agent with the power tool**

1. **Set up the agent**:
   - Use the `create_agent` function from LangChain.
   - Include the `power_tool` in the list of tools provided to the agent.
   - Use the language model defined earlier in the lab.


In [ ]:
#TODO


<details>
   <summary>Click here for Solution</summary>

```python
# List of tools for the agent
tools = [power_tool]

# Create the agent
agent = create_agent(
    model=llm,
    tools=tools
)
```

</details>


#### **Step 3: Test the agent**

1. **Test the Agent Using `invoke`**:
   - Use the `invoke` method of the agent to test its ability to calculate powers.
   - Pass a natural-language query to the agent and inspect the final message.


In [ ]:
#TODO


<details>
   <summary>Click here for Solution</summary>

```python
response = agent.invoke({
    "messages": [("human", "Calculate 5 to the power of 2.")]
})
print(response["messages"][-1].content)
```

</details>


## Authors


[Joseph Santarcangelo](https://author.skills.network/instructors/joseph_santarcangelo)


[Kunal Makwana](https://author.skills.network/instructors/kunal_makwana)


Copyright © IBM Corporation. All rights reserved.
